In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresync.gold;

CREATE OR REPLACE TABLE adb_caresync.gold.lab_test_trends AS
WITH monthly_metrics AS (
  SELECT
    test_name,
    test_date AS summary_date,
    YEAR(test_date) AS year,
    MONTH(test_date) AS month,
    DATE_TRUNC('month', test_date) AS test_start_date,
    CAST(COUNT(*) AS INT) AS total_tests,
    CAST(COUNT(*) FILTER (WHERE result_status = 'Abnormal') AS INT) AS abnormal_test_count,
    CAST(COUNT(*) FILTER (WHERE result_status = 'Critical') AS INT) AS critical_test_count
  FROM adb_caresync.silver.lab_results
  GROUP BY
    test_name,
    test_date,
    YEAR(test_date),
    MONTH(test_date),
    DATE_TRUNC('month', test_date)
),

enriched AS (
  SELECT
    mm.test_name,
    mm.year,
    mm.month,
    COALESCE(c.month_name, date_format(mm.test_start_date, 'MMMM')) AS month_name,
    CONCAT('Q', quarter(mm.test_start_date)) AS quarter_name,
    mm.total_tests,
    mm.abnormal_test_count,
    mm.critical_test_count,
    CAST(
      CASE
        WHEN mm.total_tests = 0 THEN NULL
        ELSE ROUND(mm.abnormal_test_count * 100.0 / mm.total_tests, 2)
      END AS DECIMAL(5, 2)
    ) AS abnormal_rate_pct,
    CAST(
      CASE
        WHEN mm.total_tests = 0 THEN NULL
        ELSE ROUND(mm.critical_test_count * 100.0 / mm.total_tests, 2)
      END AS DECIMAL(5, 2)
    ) AS critical_rate_pct
  FROM monthly_metrics AS mm
  LEFT JOIN adb_caresync.silver.calendar AS c
    ON CAST(date_format(mm.test_start_date, 'yyyyMMdd') AS INT) = c.date_key
)

SELECT
  test_name,
  year,
  month,
  month_name,
  quarter_name,
  total_tests,
  abnormal_test_count,
  critical_test_count,
  abnormal_rate_pct,
  critical_rate_pct,
  CAST(
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month)
    AS DECIMAL(5, 2)
  ) AS prev_month_abnormal_rate_pct,
  CAST(
    CASE
      WHEN LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) IS NULL THEN NULL
      ELSE ROUND(
        abnormal_rate_pct - LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month),
        2
      )
    END AS DECIMAL(6, 2)
  ) AS abnormal_rate_change_pct
FROM enriched;

In [0]:
updated_df = spark.table("adb_caresync.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))